In [8]:
# Cell 1 — Paths + imports
import os, json, math, re
from pathlib import Path
from typing import Dict, Any, List, Tuple, Optional

from PIL import Image, ImageDraw, ImageFont

OCR_JSON_DIR = Path("outputs\jap_ocr_otsu\json")              # your cleaned OCR JSON per page
PAGES_DIR    = Path("data/jap_imgs")                # original page PNGs
OUT_TJSON_DIR = Path("outputs/translated_json_jap")      # translated JSON per page
OUT_RENDER_DIR = Path("outputs/final_pages_bbox_jap")    # rendered pages

OUT_TJSON_DIR.mkdir(parents=True, exist_ok=True)
OUT_RENDER_DIR.mkdir(parents=True, exist_ok=True)

# Pick a font that exists on your machine (Windows examples below)
FONT_CANDIDATES = [
    r"C:\Windows\Fonts\arial.ttf",
    r"C:\Windows\Fonts\calibri.ttf",
    r"C:\Windows\Fonts\times.ttf",
]
FONT_PATH = next((p for p in FONT_CANDIDATES if os.path.exists(p)), None)
if FONT_PATH is None:
    raise FileNotFoundError("Set FONT_PATH to a valid .ttf on your system.")
print("Using font:", FONT_PATH)

Using font: C:\Windows\Fonts\arial.ttf


In [2]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [12]:
def load_json(p: Path) -> Any:
    with open(p, "r", encoding="utf-8") as f:
        return json.load(f)

def _poly_to_xyxy(poly: Any) -> List[int]:
    # poly can be [[x,y], ...] (quad or polygon)
    if not isinstance(poly, list) or len(poly) == 0:
        return [0, 0, 0, 0]
    xs, ys = [], []
    for pt in poly:
        if isinstance(pt, (list, tuple)) and len(pt) >= 2:
            xs.append(float(pt[0]))
            ys.append(float(pt[1]))
    if not xs or not ys:
        return [0, 0, 0, 0]
    return [int(min(xs)), int(min(ys)), int(max(xs)), int(max(ys))]

def _from_parallel_arrays(polys: List[Any], texts: List[Any], scores: List[Any]) -> List[Dict[str, Any]]:
    n = min(len(polys), len(texts), len(scores) if scores is not None else len(texts))
    items = []
    for i in range(n):
        poly = polys[i]
        txt = "" if texts[i] is None else str(texts[i])
        conf = None
        if scores is not None and i < len(scores):
            try:
                conf = float(scores[i]) if scores[i] is not None else None
            except Exception:
                conf = None

        items.append({
            "id": i,
            "poly": poly,
            "bbox_xyxy": _poly_to_xyxy(poly),
            "text": txt,
            "ja_text": txt,   # keeps your get_ja_text logic working
            "conf": conf
        })
    return items

def get_items(page_obj: Any) -> List[Dict[str, Any]]:
    # 1) already a list of item dicts
    if isinstance(page_obj, list):
        return page_obj

    # 2) dict-based schemas
    if isinstance(page_obj, dict):
        # old schema: {"items":[...]} or similar
        for k in ["items", "boxes", "bubbles", "ocr"]:
            if k in page_obj and isinstance(page_obj[k], list):
                return page_obj[k]

        # new compact schema: {"polys": [...], "texts": [...], "scores": [...]}
        if "polys" in page_obj and "texts" in page_obj:
            polys = list(page_obj.get("polys", []))
            texts = list(page_obj.get("texts", []))
            scores = list(page_obj.get("scores", [])) if "scores" in page_obj else None
            return _from_parallel_arrays(polys, texts, scores)

        # raw Paddle-style schema if present: {"dt_polys","rec_texts","rec_scores"}
        if "dt_polys" in page_obj and "rec_texts" in page_obj:
            polys = list(page_obj.get("dt_polys", []))
            texts = list(page_obj.get("rec_texts", []))
            scores = list(page_obj.get("rec_scores", [])) if "rec_scores" in page_obj else None
            return _from_parallel_arrays(polys, texts, scores)

    raise ValueError("Unrecognized OCR JSON schema")

In [4]:
def normalize_text(text: Optional[str]) -> str:
    # ✅ None-safe
    if text is None:
        return ""
    text = str(text)

    # basic cleanup (tweak as you like)
    text = text.replace("\u3000", " ")          # full-width space -> space
    text = text.replace("\n", " ").replace("\r", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [5]:
def get_ja_text(it: Dict[str, Any]) -> str:
    for k in ["ja_text", "text", "ocr_text", "jp_text"]:
        if k in it:
            return normalize_text(it[k])
    return ""

def get_conf(it: Dict[str, Any]) -> Optional[float]:
    for k in ["conf", "avg_conf", "confidence", "ocr_conf"]:
        if k in it:
            try:
                return float(it[k])
            except:
                return None
    return None

In [6]:
from transformers import M2M100ForConditionalGeneration, M2M100Tokenizer

model = M2M100ForConditionalGeneration.from_pretrained("facebook/m2m100_418M")
tokenizer = M2M100Tokenizer.from_pretrained("facebook/m2m100_418M")
tokenizer.src_lang = "ja"

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

In [7]:
model = model.to(device)

In [19]:
def translate_ja_to_en_fb(text: Optional[str], max_in_len: int = 256) -> str:
    text = normalize_text(text)
    if not text:
        return ""
    if tokenizer is None or model is None: # translator_mode == "placeholder" or 
        return f"[EN] {text}"

    import torch
    with torch.no_grad():
        batch = tokenizer(
            text,
            return_tensors="pt"
        ).to(device)

        gen = model.generate(**batch, forced_bos_token_id=tokenizer.get_lang_id("en"))
        out = tokenizer.decode(gen, skip_special_tokens=True)[0]

    return normalize_text(out)

In [13]:
obj = load_json(OCR_JSON_DIR / "pg_1.json")

In [14]:
items = get_items(obj)

In [20]:
out_items = []
for it in items:
    ja = get_ja_text(it)
    en = translate_ja_to_en_fb(ja) if ja else ""
    new_it = dict(it)
    new_it["ja_text"] = ja
    new_it["en_text"] = en
    c = get_conf(it)
    if c is not None:
        new_it["conf"] = c
    out_items.append(new_it)

In [21]:
out_items

[{'id': 0,
  'poly': [[830, 32], [961, 32], [961, 144], [830, 144]],
  'bbox_xyxy': [830, 32, 961, 144],
  'text': 'こ小さな港村だ',
  'ja_text': 'こ小さな港村だ',
  'conf': 0.7807550430297852,
  'en_text': 'This is a small port village.'},
 {'id': 1,
  'poly': [[821, 638], [978, 638], [978, 762], [821, 762]],
  'bbox_xyxy': [821, 638, 978, 762],
  'text': '巷には年程*元前ら[*元]',
  'ja_text': '巷には年程*元前ら[*元]',
  'conf': 0.6670724302530289,
  'en_text': 'On the street, years before.'},
 {'id': 2,
  'poly': [[821, 1172], [876, 1172], [876, 1302], [821, 1302]],
  'bbox_xyxy': [821, 1172, 876, 1302],
  'text': '風は東',
  'ja_text': '風は東',
  'conf': 0.9115988612174988,
  'en_text': 'The wind is east.'},
 {'id': 3,
  'poly': [[462, 1141], [551, 1141], [551, 1322], [462, 1322]],
  'bbox_xyxy': [462, 1141, 551, 1322],
  'text': 'おいルフイ何する気だ',
  'ja_text': 'おいルフイ何する気だ',
  'conf': 0.9461493194103241,
  'en_text': 'What do I want to do, Ruth?'},
 {'id': 4,
  'poly': [[120, 1399], [200, 1399], [200, 1497], [120, 1497]],
 

In [22]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("Helsinki-NLP/opus-mt-ja-en")
model = AutoModelForSeq2SeqLM.from_pretrained("Helsinki-NLP/opus-mt-ja-en")

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/303M [00:00<?, ?B/s]

In [23]:
model = model.to(device)

In [24]:
def translate_ja_to_en_hl(text: Optional[str], max_in_len: int = 256) -> str:
    text = normalize_text(text)
    if not text:
        return ""
    if tokenizer is None or model is None: # translator_mode == "placeholder" or 
        return f"[EN] {text}"

    import torch
    with torch.no_grad():
        batch = tokenizer(
            [text],
            return_tensors="pt"
        ).to(device)

        gen = model.generate(**batch)
        out = tokenizer.decode(gen, skip_special_tokens=True)[0]

    return normalize_text(out)

In [25]:
obj = load_json(OCR_JSON_DIR / "pg_1.json")
items = get_items(obj)
out_items = []
for it in items:
    ja = get_ja_text(it)
    en = translate_ja_to_en_hl(ja) if ja else ""
    new_it = dict(it)
    new_it["ja_text"] = ja
    new_it["en_text"] = en
    c = get_conf(it)
    if c is not None:
        new_it["conf"] = c
    out_items.append(new_it)
out_items

[{'id': 0,
  'poly': [[830, 32], [961, 32], [961, 144], [830, 144]],
  'bbox_xyxy': [830, 32, 961, 144],
  'text': 'こ小さな港村だ',
  'ja_text': 'こ小さな港村だ',
  'conf': 0.7807550430297852,
  'en_text': 'This is a small port village.'},
 {'id': 1,
  'poly': [[821, 638], [978, 638], [978, 762], [821, 762]],
  'bbox_xyxy': [821, 638, 978, 762],
  'text': '巷には年程*元前ら[*元]',
  'ja_text': '巷には年程*元前ら[*元]',
  'conf': 0.6670724302530289,
  'en_text': "I don't know what I'm talking about."},
 {'id': 2,
  'poly': [[821, 1172], [876, 1172], [876, 1302], [821, 1302]],
  'bbox_xyxy': [821, 1172, 876, 1302],
  'text': '風は東',
  'ja_text': '風は東',
  'conf': 0.9115988612174988,
  'en_text': 'Wind is east.'},
 {'id': 3,
  'poly': [[462, 1141], [551, 1141], [551, 1322], [462, 1322]],
  'bbox_xyxy': [462, 1141, 551, 1322],
  'text': 'おいルフイ何する気だ',
  'ja_text': 'おいルフイ何する気だ',
  'conf': 0.9461493194103241,
  'en_text': 'What the hell are you doing?'},
 {'id': 4,
  'poly': [[120, 1399], [200, 1399], [200, 1497], [120, 1497

In [26]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("facebook/nllb-200-distilled-600M")
model = AutoModelForSeq2SeqLM.from_pretrained("facebook/nllb-200-distilled-600M")

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

In [27]:
model = model.to(device)

In [28]:
def translate_ja_to_en_nllb(text: Optional[str], max_in_len: int = 256) -> str:
    text = normalize_text(text)
    if not text:
        return ""
    if tokenizer is None or model is None: # translator_mode == "placeholder" or 
        return f"[EN] {text}"

    import torch
    with torch.no_grad():
        batch = tokenizer(
            text,
            return_tensors="pt"
        ).to(device)

        gen = model.generate(
            **batch,
            forced_bos_token_id=tokenizer.convert_tokens_to_ids("eng_Latn"),
            max_new_tokens=256
        )        
        out = tokenizer.decode(gen, skip_special_tokens=True)[0]

    return normalize_text(out)

In [29]:
obj = load_json(OCR_JSON_DIR / "pg_1.json")
items = get_items(obj)
out_items = []
for it in items:
    ja = get_ja_text(it)
    en = translate_ja_to_en_nllb(ja) if ja else ""
    new_it = dict(it)
    new_it["ja_text"] = ja
    new_it["en_text"] = en
    c = get_conf(it)
    if c is not None:
        new_it["conf"] = c
    out_items.append(new_it)
out_items

Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'id': 0,
  'poly': [[830, 32], [961, 32], [961, 144], [830, 144]],
  'bbox_xyxy': [830, 32, 961, 144],
  'text': 'こ小さな港村だ',
  'ja_text': 'こ小さな港村だ',
  'conf': 0.7807550430297852,
  'en_text': "It's a small harbour village."},
 {'id': 1,
  'poly': [[821, 638], [978, 638], [978, 762], [821, 762]],
  'bbox_xyxy': [821, 638, 978, 762],
  'text': '巷には年程*元前ら[*元]',
  'ja_text': '巷には年程*元前ら[*元]',
  'conf': 0.6670724302530289,
  'en_text': "has a year's worth of money."},
 {'id': 2,
  'poly': [[821, 1172], [876, 1172], [876, 1302], [821, 1302]],
  'bbox_xyxy': [821, 1172, 876, 1302],
  'text': '風は東',
  'ja_text': '風は東',
  'conf': 0.9115988612174988,
  'en_text': 'The wind is from the east.'},
 {'id': 3,
  'poly': [[462, 1141], [551, 1141], [551, 1322], [462, 1322]],
  'bbox_xyxy': [462, 1141, 551, 1322],
  'text': 'おいルフイ何する気だ',
  'ja_text': 'おいルフイ何する気だ',
  'conf': 0.9461493194103241,
  'en_text': 'What are you going to do?'},
 {'id': 4,
  'poly': [[120, 1399], [200, 1399], [200, 1497], [120, 14